# Langkah 4 — Bangun Gold Standard untuk Validasi
Membuat sampel terstratefikasi ~200 ulasan dan menulis template anotasi untuk **tiga anotator** (format Excel `.xlsx`).

Tujuan: mengukur seberapa andal ekstraksi LLM dengan angka yang bisa dipertanggungjawabkan
(precision/recall/F1 per kategori + kesepakatan antar-anotator Cohen's κ).

In [1]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
DATA_DIR = ROOT.parent / 'data'
OUT = ROOT / 'outputs'
sys.path.insert(0, str(ROOT))

import pandas as pd
from absa.goldset import build_sample, write_template, ANNOTATION_COLUMNS

df = pd.read_csv(DATA_DIR / 'reviews_cleaned_rating_1_2.csv')
print(f'Dataset lengkap: {len(df)} ulasan')
print(f'Kategori anotasi: {ANNOTATION_COLUMNS}')

Dataset lengkap: 8641 ulasan
Kategori anotasi: ['Responsiveness', 'Reliability', 'Assurance', 'Empathy', 'Tangibles', 'Umum']


## 1. Bangun sampel terstratefikasi
Mengover-sampling wilayah kecil, ulasan pendek, dan rating 2★ — bagian yang paling sulit dan paling sering salah.

In [2]:
sample = build_sample(df, n_total=200, seed=42)
print(f'Ukuran sampel: {len(sample)}')
print('\nSebaran per stratum:')
print(sample['stratum'].value_counts().sort_index().to_string())
print('\nKeseimbangan wilayah:', sample['wilayah'].value_counts().to_dict())
print('Keseimbangan rating :', sample['rating'].value_counts().to_dict())

Ukuran sampel: 200

Sebaran per stratum:
stratum
Bantul | panjang | 1★      13
Bantul | panjang | 2★       6
Bantul | pendek | 1★        8
Bantul | pendek | 2★        2
Bantul | sedang | 1★       13
Bantul | sedang | 2★        6
Semarang | panjang | 1★    15
Semarang | panjang | 2★     7
Semarang | pendek | 1★      8
Semarang | pendek | 2★      3
Semarang | sedang | 1★     14
Semarang | sedang | 2★      7
Surabaya | panjang | 1★    30
Surabaya | panjang | 2★    11
Surabaya | pendek | 1★     14
Surabaya | pendek | 2★      5
Surabaya | sedang | 1★     28
Surabaya | sedang | 2★     10

Keseimbangan wilayah: {'Surabaya': 98, 'Semarang': 54, 'Bantul': 48}
Keseimbangan rating : {1: 143, 2: 57}


## 2. Tulis template anotasi (tiga salinan, format Excel)
Setiap anotator mengisi salinannya **secara mandiri**, tanpa melihat hasil anotator lain maupun output model.
File `.xlsx` bisa langsung dibuka di Excel — kolom ulasan lebar, kolom kategori sempit agar mudah diisi.

In [3]:
# Simpan daftar review_id sampel agar evaluasi nanti memakai sampel yang sama persis
sample[['review_id','puskesmas_id','puskesmas_name','wilayah','rating','stratum','review_text']].to_csv(
    OUT / 'goldset_sample.csv', index=False, encoding='utf-8-sig')

# Tiga template kosong, satu per anotator
write_template(sample, OUT / 'goldset_anotasi_1.xlsx')
write_template(sample, OUT / 'goldset_anotasi_2.xlsx')
write_template(sample, OUT / 'goldset_anotasi_3.xlsx')
print('Tertulis:')
print('  outputs/goldset_sample.csv      (acuan sampel)')
print('  outputs/goldset_anotasi_1.xlsx  (untuk anotator 1)')
print('  outputs/goldset_anotasi_2.xlsx  (untuk anotator 2)')
print('  outputs/goldset_anotasi_3.xlsx  (untuk anotator 3)')

Tertulis:
  outputs/goldset_sample.csv      (acuan sampel)
  outputs/goldset_anotasi_1.xlsx  (untuk anotator 1)
  outputs/goldset_anotasi_2.xlsx  (untuk anotator 2)
  outputs/goldset_anotasi_3.xlsx  (untuk anotator 3)


## 3. Panduan Anotasi (WAJIB dibaca ketiga anotator)

Konsistensi panduan ini menentukan tinggi-rendahnya Cohen's κ. Jika κ rendah, **rubriknya** yang ambigu, bukan modelnya.

### Cara mengisi
Untuk **setiap ulasan**, isi tiap kolom kategori dengan salah satu:
- `neg` — kategori ini disebut sebagai **keluhan/masalah**
- `pos` — kategori ini disebut sebagai **pujian**
- `both` — kategori yang sama disebut **positif sekaligus negatif** dalam satu ulasan
- *(kosong)* — kategori ini **tidak disebut** sama sekali

Satu ulasan boleh mengisi beberapa kolom (mis. antri lama + petugas judes → `Responsiveness=neg`, `Empathy=neg`).

> **Prinsip utama:** beri label sesuai **isi teks yang benar-benar ada**, bukan menebak jawaban model. Anotasi dan model dibandingkan belakangan.

### Definisi kategori
| Kategori | Cakupan | Contoh pemicu |
|---|---|---|
| **Responsiveness** | Kecepatan & waktu tunggu, antrean, diabaikan | antri lama, lelet, dibiarkan nunggu |
| **Reliability** | Obat, BPJS, rujukan, jam buka, konsistensi/janji | obat kosong, BPJS dipersulit, tutup padahal jam buka |
| **Assurance** | Kompetensi, diagnosis, profesionalisme, keamanan | salah diagnosis, asal periksa, tambah parah |
| **Empathy** | Sikap, keramahan, komunikasi, perhatian | judes, tidak dijelaskan, pilih kasih |
| **Tangibles** | Kebersihan, kenyamanan, fasilitas fisik | kotor, panas, AC mati, parkir susah |
| **Umum** | Sentimen umum **tanpa** aspek spesifik | "pelayanan buruk", "kecewa", "kapok" |

### Aturan batas (kasus yang sering ambigu — sepakati di sini)
1. **Responsiveness vs Reliability** — soal *kecepatan/menunggu* → Responsiveness; soal *ketersediaan/administrasi/janji ditepati* (obat, BPJS, rujukan, jam buka) → Reliability.
   - Pendaftaran online: keluhan *"lemot/error"* → Responsiveness; keluhan *"data hilang/sistem tak bisa diandalkan"* → Reliability.
2. **Empathy vs Assurance** — soal *sikap/keramahan/komunikasi* → Empathy; soal *kemampuan medis/diagnosis/profesionalisme* → Assurance.
   - "Dokter judes" → Empathy. "Dokter salah diagnosis" → Assurance. Bisa keduanya jika teks menyebut dua-duanya.
3. **Umum hanya pilihan terakhir** — jika ada aspek spesifik yang disebut, pakai dimensi itu, **jangan** Umum. Umum hanya untuk "pelayanan jelek" tanpa rincian apa pun.
4. **Sarkasme** — pada ulasan 1–2★, pujian yang jelas menyindir ("mantap pelayanannya") dihitung `neg`.
5. Abaikan basa-basi murni (doa, salam penutup surat) → biarkan semua kolom kosong.

## 4. Setelah ketiga anotator selesai
Lanjut ke `03_evaluate_goldset.ipynb` untuk:
1. Hitung Cohen's κ antar-anotator secara berpasangan (cek konsistensi rubrik),
2. Adjudikasi — **majority vote** (2 dari 3 sepakat = gold), hanya ketidaksepakatan 3 arah yang diselesaikan manual,
3. Skor model vs gold (precision/recall/F1 + CI) dan taksonomi error.